# 02 — Базовая модель и сравнение

**Цель ноутбука:** обучить baseline-модель (LogisticRegression) и сравнить её
с ансамблями (RandomForest, GradientBoosting) по кросс-валидации.

**Экспериментальный протокол:** стратифицированное разбиение train/test,
5-fold кросс-валидация на обучающей выборке, метрики по out-of-fold предсказаниям.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd

from src.data.loader import get_dataset, split_data
from src.models.registry import build_model, build_registry
from src.models.trainer import (
    build_pipeline, cross_validate_model, run_experiments, results_to_dataframe,
)

pd.set_option("display.max_columns", None)

## 1. Данные и разбиение

In [ ]:
df = get_dataset(n_customers=7000, random_state=42)
X_train, X_test, y_train, y_test = split_data(df, test_size=0.2, random_state=42)
print(f"train: {len(X_train)}, test: {len(X_test)}")
print(f"доля оттока — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

## 2. Baseline — LogisticRegression

Линейная модель с `class_weight="balanced"` — простая, быстрая,
интерпретируемая стартовая точка.

In [ ]:
baseline = build_pipeline(build_model("logistic_regression", {
    "C": 1.0, "max_iter": 1000, "class_weight": "balanced", "random_state": 42,
}))
cv_metrics = cross_validate_model(baseline, X_train, y_train, n_splits=5)
print("Baseline — метрики 5-fold кросс-валидации:")
for k, v in cv_metrics.items():
    print(f"  {k:12s} {v:.4f}")

## 3. Сравнение с ансамблями

Запускаем единый протокол для всех трёх моделей-кандидатов из конфига.

In [ ]:
registry = build_registry({
    "baseline": {"type": "logistic_regression", "enabled": True,
                 "params": {"C": 1.0, "max_iter": 1000,
                            "class_weight": "balanced", "random_state": 42}},
    "random_forest": {"type": "random_forest", "enabled": True,
                      "params": {"n_estimators": 300, "max_depth": 12,
                                 "min_samples_leaf": 20, "class_weight": "balanced",
                                 "n_jobs": -1, "random_state": 42}},
    "gradient_boosting": {"type": "gradient_boosting", "enabled": True,
                          "params": {"n_estimators": 250, "learning_rate": 0.05,
                                     "max_depth": 3, "subsample": 0.9,
                                     "random_state": 42}},
})

results = run_experiments(registry, X_train, y_train, X_test, y_test, cv_splits=5)
comparison = results_to_dataframe(results)
comparison

## 4. Выводы

- **GradientBoosting** лидирует по ROC-AUC и PR-AUC на кросс-валидации.
- **LogisticRegression** — сильный baseline, лучший по recall и F1 при пороге 0.5.
- **RandomForest** уступает обоим конкурентам.

Подробный анализ и финальный выбор — в ноутбуке `03_models_tuning.ipynb`.
Итоговое обоснование — в `report.md`, раздел 5.